# TextCNN 与双向 LSTM 的互补性与融合

使用相同验证集和序列处理规则，对比种子 42 的 TextCNN 与双向 LSTM。
在种子 42 的验证预测上搜索融合权重，再将选定权重用于种子 123 的模型作为随机种子复验。
两个种子使用同一个验证集，因此复验不等于独立数据集上的泛化评估；测试集不参与本实验。


In [ ]:
# 加载数据、模型和检查点
import hashlib
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from sentiment_analysis.evaluation import classification_metrics
from sentiment_analysis.lstm import BidirectionalLSTMClassifier
from sentiment_analysis.paths import CHECKPOINTS_DIR, CLEANED_DIR, TABLES_DIR, FIGURES_DIR
from sentiment_analysis.sequence_data import SentimentDataset, TOKEN_PATTERN
from sentiment_analysis.textcnn import TextCNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
validation_path = CLEANED_DIR / "val.csv"
validation_data = pd.read_csv(validation_path)
validation_hash = hashlib.sha256(validation_path.read_bytes()).hexdigest()

checkpoint_paths = {
    42: {
        "TextCNN": CHECKPOINTS_DIR / "textcnn_extended_best.pt",
        "BiLSTM": CHECKPOINTS_DIR / "lstm_bidirectional_seed42_best.pt",
    },
    123: {
        "TextCNN": CHECKPOINTS_DIR / "textcnn_kernel3_seed123.pt",
        "BiLSTM": CHECKPOINTS_DIR / "lstm_bidirectional_seed123_best.pt",
    },
}

print(f"Device: {device} | Validation: {len(validation_data):,}")
print(f"Validation SHA-256: {validation_hash}")


In [ ]:
# 检查配置并生成每对模型的验证概率
def predict_probabilities(model, data_loader):
    model.eval()
    labels_all = []
    probabilities_all = []

    with torch.no_grad():
        for token_ids, lengths, labels in data_loader:
            logits = model(token_ids.to(device), lengths)
            probabilities = torch.softmax(logits, dim=1)[:, 1]
            labels_all.extend(labels.tolist())
            probabilities_all.extend(probabilities.cpu().tolist())

    return labels_all, np.array(probabilities_all)


seed_predictions = {}
checkpoint_metadata = {}

for seed, paths in checkpoint_paths.items():
    textcnn_checkpoint = torch.load(paths["TextCNN"], map_location=device, weights_only=True)
    lstm_checkpoint = torch.load(paths["BiLSTM"], map_location=device, weights_only=True)

    for checkpoint in [textcnn_checkpoint, lstm_checkpoint]:
        if checkpoint["token_pattern"] != TOKEN_PATTERN.pattern or not checkpoint["lowercase"]:
            raise ValueError("Checkpoint tokenizer does not match current sequence processing.")
        if checkpoint["random_seed"] != seed:
            raise ValueError("Checkpoint seed does not match the experiment.")

    if (textcnn_checkpoint["vocabulary"] != lstm_checkpoint["vocabulary"]
            or textcnn_checkpoint["max_length"] != lstm_checkpoint["max_length"]):
        raise ValueError("TextCNN and BiLSTM sequence processing does not match.")

    validation_dataset = SentimentDataset(
        validation_data, textcnn_checkpoint["vocabulary"], textcnn_checkpoint["max_length"],
    )
    validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)

    textcnn_model = TextCNN(**textcnn_checkpoint["model_config"]).to(device)
    textcnn_model.load_state_dict(textcnn_checkpoint["model_state_dict"])
    lstm_model = BidirectionalLSTMClassifier(**lstm_checkpoint["model_config"]).to(device)
    lstm_model.load_state_dict(lstm_checkpoint["model_state_dict"])

    textcnn_labels, textcnn_probabilities = predict_probabilities(textcnn_model, validation_loader)
    lstm_labels, lstm_probabilities = predict_probabilities(lstm_model, validation_loader)

    if textcnn_labels != lstm_labels or textcnn_labels != validation_data["label"].tolist():
        raise ValueError("Validation labels are not aligned.")

    seed_predictions[seed] = {
        "labels": np.array(textcnn_labels),
        "TextCNN": textcnn_probabilities,
        "BiLSTM": lstm_probabilities,
    }
    checkpoint_metadata[seed] = {
        "textcnn_file": paths["TextCNN"].name,
        "textcnn_sha256": hashlib.sha256(paths["TextCNN"].read_bytes()).hexdigest(),
        "textcnn_epoch": textcnn_checkpoint["epoch"],
        "bilstm_file": paths["BiLSTM"].name,
        "bilstm_sha256": hashlib.sha256(paths["BiLSTM"].read_bytes()).hexdigest(),
        "bilstm_epoch": lstm_checkpoint["epoch"],
    }
    print(f"Seed {seed}: CNN epoch {textcnn_checkpoint['epoch']}, BiLSTM epoch {lstm_checkpoint['epoch']}")


## 预测互补性

先观察种子 42 下两个模型分别正确、共同错误的情况。样本中的差异表明两个模型预测不完全相同，但不保证融合一定改善。


In [ ]:
# 比较两个模型的预测是否正确
prediction_comparison = validation_data[["text", "label"]].copy()
prediction_comparison["TextCNN"] = (seed_predictions[42]["TextCNN"] > 0.5).astype(int)
prediction_comparison["BiLSTM"] = (seed_predictions[42]["BiLSTM"] > 0.5).astype(int)

textcnn_correct = prediction_comparison["TextCNN"] == prediction_comparison["label"]
bilstm_correct = prediction_comparison["BiLSTM"] == prediction_comparison["label"]

comparison_summary = pd.DataFrame([
    {"CASE": "Both correct", "COUNT": (textcnn_correct & bilstm_correct).sum()},
    {"CASE": "Only TextCNN correct", "COUNT": (textcnn_correct & ~bilstm_correct).sum()},
    {"CASE": "Only BiLSTM correct", "COUNT": (~textcnn_correct & bilstm_correct).sum()},
    {"CASE": "Both wrong", "COUNT": (~textcnn_correct & ~bilstm_correct).sum()},
])

display(comparison_summary)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
comparison_summary.to_csv(TABLES_DIR / "textcnn_bilstm_prediction_comparison.csv", index=False)


In [ ]:
# 查看预测分歧及共同误判的评论
for name, mask in [
    ("Only TextCNN correct", textcnn_correct & ~bilstm_correct),
    ("Only BiLSTM correct", ~textcnn_correct & bilstm_correct),
    ("Both wrong", ~textcnn_correct & ~bilstm_correct),
]:
    print(name)
    display(prediction_comparison[mask].head(3))


## 软投票权重对照

将两个模型的正类概率加权平均，超过 0.5 时预测为正类。先在种子 42 上以 0.1 为间隔搜索权重，以验证集 F1 选取；同分时优先接近等权重，再选 CNN 权重较小者。


In [ ]:
# 在种子 42 的验证集上比较权重
labels = seed_predictions[42]["labels"]
textcnn_probabilities = seed_predictions[42]["TextCNN"]
bilstm_probabilities = seed_predictions[42]["BiLSTM"]

weight_results = []
for step in range(11):
    textcnn_weight = step / 10
    ensemble_probabilities = textcnn_weight * textcnn_probabilities + (1 - textcnn_weight) * bilstm_probabilities
    predictions = (ensemble_probabilities > 0.5).astype(int)

    weight_results.append({
        "TEXTCNN WEIGHT": textcnn_weight,
        "BILSTM WEIGHT": 1 - textcnn_weight,
        **classification_metrics(labels, predictions),
    })

weight_comparison = pd.DataFrame(weight_results)
best_candidates = weight_comparison[weight_comparison["F1"] == weight_comparison["F1"].max()].copy()
best_candidates["DISTANCE TO EQUAL"] = (best_candidates["TEXTCNN WEIGHT"] - 0.5).abs()
best_weight_result = best_candidates.sort_values(["DISTANCE TO EQUAL", "TEXTCNN WEIGHT"]).iloc[0]
selected_weight = float(best_weight_result["TEXTCNN WEIGHT"])

display(weight_comparison.round(4))
print(f"Selected weights: TextCNN {selected_weight:.1f}, BiLSTM {1-selected_weight:.1f}")
weight_comparison.to_csv(TABLES_DIR / "textcnn_bilstm_ensemble_weights.csv", index=False)


In [ ]:
# 比较单模型、等权重和固定的选定权重
results = []
for seed in [42, 123]:
    labels = seed_predictions[seed]["labels"]
    textcnn_probabilities = seed_predictions[seed]["TextCNN"]
    bilstm_probabilities = seed_predictions[seed]["BiLSTM"]

    for name, textcnn_weight in [
        ("TextCNN", 1.0),
        ("BiLSTM", 0.0),
        ("Equal-weight ensemble", 0.5),
        ("Selected-weight ensemble", selected_weight),
    ]:
        probabilities = textcnn_weight * textcnn_probabilities + (1-textcnn_weight) * bilstm_probabilities
        predictions = (probabilities > 0.5).astype(int)
        results.append({
            "SEED": seed,
            "MODEL": name,
            "TEXTCNN WEIGHT": textcnn_weight,
            "BILSTM WEIGHT": 1-textcnn_weight,
            **classification_metrics(labels, predictions),
        })

ensemble_comparison = pd.DataFrame(results)
display(ensemble_comparison.round(4))
ensemble_comparison.to_csv(TABLES_DIR / "textcnn_bilstm_ensemble_comparison.csv", index=False)

# 上一轮使用单向 LSTM 的等权融合结果
previous = pd.read_csv(TABLES_DIR / "textcnn_lstm_ensemble_comparison.csv")
print("Previous unidirectional LSTM equal-weight ensemble:")
display(previous[previous["MODEL"] == "Equal-weight ensemble"].round(4))


In [ ]:
# 统计选定权重融合相对于两个单模型纠正和新增的错误
selected_probabilities = (
    selected_weight * seed_predictions[42]["TextCNN"]
    + (1 - selected_weight) * seed_predictions[42]["BiLSTM"]
)
prediction_comparison["Ensemble"] = (selected_probabilities > 0.5).astype(int)
ensemble_correct = prediction_comparison["Ensemble"] == prediction_comparison["label"]

error_changes = pd.DataFrame([
    {"MODEL": "TextCNN", "CORRECTED": (~textcnn_correct & ensemble_correct).sum(),
     "NEW ERRORS": (textcnn_correct & ~ensemble_correct).sum()},
    {"MODEL": "BiLSTM", "CORRECTED": (~bilstm_correct & ensemble_correct).sum(),
     "NEW ERRORS": (bilstm_correct & ~ensemble_correct).sum()},
])
display(error_changes)
error_changes.to_csv(TABLES_DIR / "textcnn_bilstm_ensemble_error_changes.csv", index=False)


In [ ]:
# 保存验证概率与数据、检查点的版本信息
prediction_rows = []
for seed in [42, 123]:
    for index, (label, textcnn_probability, bilstm_probability) in enumerate(zip(
        seed_predictions[seed]["labels"],
        seed_predictions[seed]["TextCNN"],
        seed_predictions[seed]["BiLSTM"],
    )):
        prediction_rows.append({
            "SEED": seed,
            "VALIDATION ROW": index,
            "LABEL": label,
            "TEXTCNN PROBABILITY": textcnn_probability,
            "BILSTM PROBABILITY": bilstm_probability,
        })

pd.DataFrame(prediction_rows).to_csv(
    TABLES_DIR / "textcnn_bilstm_validation_probabilities.csv", index=False,
)

metadata = {
    "validation_sha256": validation_hash,
    "validation_rows": len(validation_data),
    "checkpoint_files": checkpoint_metadata,
    "selected_textcnn_weight": selected_weight,
    "selection_seed": 42,
    "threshold": 0.5,
}
with (TABLES_DIR / "textcnn_bilstm_ensemble_metadata.json").open("w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)


In [ ]:
# 绘制种子 42 的权重与验证 F1 关系
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(8, 5))
axis.plot(weight_comparison["TEXTCNN WEIGHT"], weight_comparison["F1"], marker="o")
axis.set_xlabel("TextCNN weight (BiLSTM weight = 1 - TextCNN weight)")
axis.set_ylabel("Validation F1")
axis.set_title("TextCNN + BiLSTM ensemble (seed 42)")
axis.set_xticks(weight_comparison["TEXTCNN WEIGHT"])
axis.set_ylim(0, 1)
axis.grid(alpha=0.3)
figure.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
figure.savefig(FIGURES_DIR / "textcnn_bilstm_ensemble_weights.png", dpi=150)
plt.show()


## 本次实验的观察

- 种子 42 下，CNN 与双向 LSTM 分别有 219 和 166 条只有自己判断正确的评论，另有 163 条共同误判。
- 种子 42 下，等权重融合的 F1 为 0.8788；以 0.1 为步长搜索时，CNN 0.7、双向 LSTM 0.3 的 F1 为 0.8798。
- 将 0.7 / 0.3 固定用于种子 123 时，F1 为 0.8696，低于该种子的等权重融合 F1 0.8731。
- 此前种子 42 的 CNN 与单向 LSTM 等权融合 F1 为 0.8828，高于本次对应的双向 LSTM 等权融合结果。两种比较仍然只针对当前检查点，不能据此断言结构的普遍优劣。

权重在验证集上选择，而两组种子使用同一个验证集；这些结果是探索性结果，不能替代独立测试集评估。